## Vector database and embeddings

### Install libraries

In [1]:
!pip install faiss-cpu sentence_transformers

Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

### Generate embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

model.max_seq_length = 256

sentences = [
    "dinosaurs live in africa but in different time dimension", 
    "this is sentence about little cat that liked this eat fast food",
    "this is the another sample sentence which is here just this not be matched while other one is"
]

embeddings = model.encode(sentences, normalize_embeddings=True)

In [2]:
embeddings

array([[-0.04987684,  0.03634832,  0.01747593, ..., -0.05154557,
         0.01327896, -0.05160031],
       [ 0.07967843,  0.06043371,  0.0579344 , ...,  0.14950444,
         0.06014551,  0.02409914],
       [-0.008951  ,  0.05474586,  0.08349688, ...,  0.00953911,
         0.09756645,  0.00701754]], dtype=float32)

In [4]:
print(len(embeddings[1]))

384


### Create vector database and load documents

In [5]:
import faiss

d = 384  # number of dimensions

index = faiss.IndexFlatL2(d)  #  build the index
index.add(embeddings)  #  add vectors to the index

### Search the vector database

In [18]:
queryText = "french fries"
embeddingSearch = model.encode([queryText], normalize_embeddings=True)
embeddingFound, idx = index.search(embeddingSearch, 1)  #  actual search
print(queryText + " matches:\\n" + sentences[idx[0][0]])

queryText = "not similar text"
embeddingSearch = model.encode([queryText], normalize_embeddings=True)
embeddingFound, idx = index.search(embeddingSearch, 1)  #  actual search
print(queryText + " matches:\\n" + sentences[idx[0][0]])

french fries matches:\nthis is sentence about little cat that liked this eat fast food
not similar text matches:\nthis is the another sample sentence which is here just this not be matched while other one is


###  Agent using Vector Database
The following example agent gets a retriever tool. Just like the desks_free and room_status tools earlier, the model decides when to call 
it ; here, when a question needs looking something up in the knowledge base. I used InMemoryVectorStore because it needs no extra 
database install; it's the teaching-friendly stand-in for a real vector DB.

In [22]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.tools.retriever import create_retriever_tool
from langchain.agents import create_agent

# 1. Some documents to act as our knowledge base
docs = [
    Document(page_content="SpiralTrain is an IT training company based in Houten, Netherlands."),
    Document(page_content="The LangChain course covers chains, agents, tools, and memory."),
    Document(page_content="SpiralTrain courses can be delivered on-site or online."),
    Document(page_content="The Python course lasts 3 days and covers the language fundamentals."),
]

# 2. Embed the documents and store them in a vector store
embeddings = make_embeddings()
vector_store = InMemoryVectorStore.from_documents(docs, embedding=embeddings)

# 3. Turn the store into a retriever, then wrap it as a tool
retriever = vector_store.as_retriever(search_kwargs={"k": 2})  # return top 2 matches
retriever_tool = create_retriever_tool(
    retriever,
    name="search_courses",
    description="Search information about SpiralTrain and its IT courses.",
)

# 4. Build the agent with the retriever tool
llm = make_llm()
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt="You are a helpful assistant. Use the search tool to answer questions about SpiralTrain.",
)

# 5. Ask something that requires looking in the knowledge base
#result = agent.invoke({"messages": [("human", "Where is SpiralTrain based and what does the LangChain course cover?")]})
result = agent.invoke({"messages": [("human", "How long does the Python course lasts?")]})
print(result["messages"][-1].content)

The Python course at SpiralTrain lasts for 3 days and covers the fundamentals of the language.


### Vector database on disk

In [7]:
# sqlite-vss ships wheels for Linux and macOS only, so this section runs in
# Colab but not on a Windows laptop. Everything above it is unaffected.
!pip install sentence-transformers sqlite-vss

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [8]:
import sqlite3
import json
import numpy as np
from sentence_transformers import SentenceTransformer
import sqlite_vss

DB_PATH = "vectors.db"
DIM = 384
model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

def embed_norm(texts):
    v = model.encode(texts).astype("float32")
    v /= (np.linalg.norm(v, axis=1, keepdims=True) + 1e-12)  #  kosinus → IP
    return v

con = sqlite3.connect(DB_PATH)
con.enable_load_extension(True)
sqlite_vss.load(con)
cur = con.cursor()

#  1) Tabel
cur.executescript(f"""
CREATE TABLE IF NOT EXISTS docs(
  id INTEGER PRIMARY KEY,
  text TEXT NOT NULL
);
CREATE VIRTUAL TABLE IF NOT EXISTS doc_index USING vss0(emb({DIM}));
""")
con.commit()

#  2) Data + index
docs = [
    (1, "The Fibonacci sequence starts with 0 and 1."),
    (2, "Dijkstra's algorithm finds the shortest paths in a graph."),
    (3, "Recursion is a function calling itself."),
]
cur.executemany("INSERT OR IGNORE INTO docs(id, text) VALUES (?,?)", docs)

embs = embed_norm([t for _, t in docs]).tolist()

#  (a) if exist : remove old vectors by rowid
cur.executemany("DELETE FROM doc_index WHERE rowid = ?", [(d[0],) for d in docs])
#  (b) insert as raw JSON string
cur.executemany(
    "INSERT INTO doc_index(rowid, emb) VALUES (?, ?)",
    [(docs[i][0], json.dumps(embs[i])) for i in range(len(docs))]
)
con.commit()

query = "How does the Fibonacci sequence begin?"
q = embed_norm([query])[0].tolist()

rows = cur.execute("""
WITH knn AS (
  SELECT rowid, distance
  FROM doc_index
  WHERE vss_search(emb, ?)
  ORDER BY distance DESC
  LIMIT 5
)
SELECT d.id, d.text, knn.distance
FROM knn
JOIN docs AS d ON d.id = knn.rowid
ORDER BY knn.distance DESC;
""", (json.dumps(q),)).fetchall()

for rid, text, score in rows:
    print(f"id={rid} score={score:.4f}  text={text}")

id=2 score=1.6242  text=Dijkstra's algorithm finds the shortest paths in a graph.
id=3 score=1.1552  text=Recursion is a function calling itself.
id=1 score=0.3085  text=The Fibonacci sequence starts with 0 and 1.
